[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [sqlite3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)

# Adapters and Converters &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell builds `scratch/stations.db` with the stations and their year of readings, registers
the adapter and converter for `datetime` that the notebook registered, and opens a connection with
both `detect_types` flags, which the tasks share. Run it first. The tasks do not depend on one
another, and the last cell closes the connection and removes the scratch folder.


In [1]:
import math
import shutil
import sqlite3
from datetime import datetime, timedelta, timezone
from pathlib import Path

SCRATCH = Path("scratch")
SCRATCH.mkdir(exist_ok=True)
DATABASE = SCRATCH / "stations.db"
DATABASE.unlink(missing_ok=True)
STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


def adapt_datetime(value):
    """A datetime as ISO 8601 text to the minute."""
    return value.isoformat(timespec="minutes")


def convert_datetime(value):
    """ISO 8601 text, received as bytes, back into a datetime."""
    return datetime.fromisoformat(value.decode())


sqlite3.register_adapter(datetime, adapt_datetime)
sqlite3.register_converter("datetime", convert_datetime)

conn = sqlite3.connect(DATABASE, detect_types=sqlite3.PARSE_DECLTYPES | sqlite3.PARSE_COLNAMES)
conn.executescript("""
    CREATE TABLE stations (id INTEGER PRIMARY KEY, name TEXT NOT NULL, latitude REAL NOT NULL);
    CREATE TABLE readings (id INTEGER PRIMARY KEY, station_id INTEGER NOT NULL REFERENCES stations (id),
                           hour TEXT NOT NULL, celsius REAL);
""")
station_ids = {name: conn.execute("INSERT INTO stations (name, latitude) VALUES (?, ?)", (name, latitude)).lastrowid
               for name, latitude in LATITUDES.items()}
conn.executemany("INSERT INTO readings (station_id, hour, celsius) VALUES (?, ?, ?)",
                 ((station_ids[station], hour, celsius) for station, hour, celsius in year_of_readings()))
conn.commit()

print("built", DATABASE)


built scratch/stations.db


**1.** A reading whose hour is a `datetime`.


In [2]:
conn.execute("INSERT INTO readings (station_id, hour, celsius) VALUES (?, ?, ?)",
             (station_ids["Kirkenes"], datetime(2025, 12, 1, 6, 0), -9.4))
conn.commit()

print(conn.execute("SELECT hour, typeof(hour) FROM readings WHERE station_id = ?", (station_ids["Kirkenes"],)).fetchone())


('2025-12-01T06:00', 'text')


The adapter wrote the `datetime` as `'2025-12-01T06:00'`, text in the same form as every other hour,
and the query reads `hour` with no bracketed name, so it comes back as that text.


**2.** The hour of Oslo's warmest reading, as a `datetime`.


In [3]:
hour, celsius = conn.execute("""
    SELECT r.hour AS "hour [datetime]", r.celsius
    FROM readings AS r JOIN stations AS s ON s.id = r.station_id
    WHERE s.name = ? AND r.celsius IS NOT NULL
    ORDER BY r.celsius DESC, r.hour
    LIMIT 1
""", ("Oslo",)).fetchone()

print(celsius, "at", hour, "on a", hour.strftime("%A"))


19.3 at 2025-07-17 15:00:00 on a Thursday


`hour` is declared `TEXT`, so only the name in brackets could ask for the `datetime` converter, and a
real `datetime` knows its weekday.


**3.** A `timedelta` stored as whole seconds.


In [4]:
def adapt_timedelta(value):
    """A timedelta as whole seconds."""
    return int(value.total_seconds())


def convert_duration(value):
    """Whole seconds, received as bytes, back into a timedelta."""
    return timedelta(seconds=int(value))


sqlite3.register_adapter(timedelta, adapt_timedelta)
sqlite3.register_converter("duration", convert_duration)

conn.execute("CREATE TABLE visit_lengths (station_id INTEGER NOT NULL, length DURATION NOT NULL)")
conn.execute("INSERT INTO visit_lengths VALUES (?, ?)", (station_ids["Kirkenes"], timedelta(hours=4, minutes=45)))
conn.commit()

print(conn.execute("SELECT length, typeof(length) FROM visit_lengths").fetchone())


(datetime.timedelta(seconds=17100), 'integer')


The adapter stored 17100 seconds, as an integer, since `DURATION` gives the column NUMERIC affinity.
`PARSE_DECLTYPES` found the converter by the declared type's name, and `typeof` shows what SQLite
holds underneath.


**4.** Bergen's readings in every month.


In [5]:
months = conn.execute("""
    SELECT strftime('%m', r.hour) AS month, COUNT(*) AS readings
    FROM readings AS r JOIN stations AS s ON s.id = r.station_id
    WHERE s.name = ? AND r.hour >= ? AND r.hour < ?
    GROUP BY month
    ORDER BY month
""", ("Bergen", datetime(2025, 1, 1), datetime(2026, 1, 1))).fetchall()

print(months[:3])


[('01', 744), ('02', 672), ('03', 744)]


`strftime('%m', ...)` takes the month from every hour, and `GROUP BY` counts them: 31 days of 24
hours in January, 28 days in February.


**5.** Svalbard's coldest hour, as an epoch and back.


In [6]:
hour, epoch = conn.execute("""
    SELECT r.hour, CAST(strftime('%s', r.hour) AS INTEGER)
    FROM readings AS r JOIN stations AS s ON s.id = r.station_id
    WHERE s.name = ? AND r.celsius IS NOT NULL
    ORDER BY r.celsius, r.hour
    LIMIT 1
""", ("Svalbard",)).fetchone()

print(hour, "->", epoch, "->", datetime.fromtimestamp(epoch, timezone.utc))


2025-01-12T03:00 -> 1736650800 -> 2025-01-12 03:00:00+00:00


SQLite reads the stored hour as UTC, `strftime('%s', ...)` gives the epoch as text, and `CAST` makes
it an integer. `fromtimestamp` with `timezone.utc` reads it back in UTC, whatever the machine's own
time zone is.


**6.** A class that adapts itself.


In [7]:
class Coordinates:
    """A point on the Earth, in degrees."""

    def __init__(self, latitude, longitude):
        self.latitude = latitude
        self.longitude = longitude

    def __conform__(self, protocol):
        if protocol is sqlite3.PrepareProtocol:
            return f"{self.latitude};{self.longitude}"


print(conn.execute("SELECT ?, typeof(?)", (Coordinates(59.91, 10.75), Coordinates(59.91, 10.75))).fetchone())


('59.91;10.75', 'text')


With no adapter registered for `Coordinates`, sqlite3 asked the object itself, through
`__conform__`, and stored the text it returned. An adapter suits a type you cannot change, and
`__conform__` a class of your own.

Last, close the connection and remove the scratch folder:


In [8]:
conn.close()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Adapters and Converters](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/09-adapters-and-converters.ipynb)  &nbsp;&middot;&nbsp;  [sqlite3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)
